# NLP Prototype — النسخة الكاملة في notebook واحد

الـ notebook ده فيه **كل كود الـ prototype** جوه الخلايا، من غير ما يستورد من فولدر `nlp_proto/`. تقدر تقرا كل سطر وتعدّل فيه وتشغّل.

**المطلوب:** يتشغّل من جوه فولدر `nlp_prototype` (عشان يلاقي `data/` و`models/`)، وتكون مثبّت `spacy` و`scikit-learn` و`pandas`.

**ملخص:** مفيش fine-tuning لموديل جاهز. الموديلات (الـ Router، وTF-IDF + Logistic Regression، وspaCy NER) **متدربة من الصفر** على داتا اصطناعية كتبها Gemini 4 Argon، والمبلغ بقواعد. الـ LLM اختياري، وبيُستخدم بس للنصايح وشرح أسئلة الأسواق لو `GEMINI_API_KEY` متحدد.

In [1]:
import os
print('working folder:', os.getcwd())
assert os.path.isdir('data/train'), 'شغّل الـ notebook من جوه فولدر nlp_prototype'

working folder: D:\ai-financial-advisor-team\ai-orchestrator\nlp_prototype


## 1) توحيد النص — `normalize.py`
بيحوّل الأرقام العربي لإنجليزي ويوحّد الحروف (أ/إ/آ ← ا، ة ← ه، ى ← ي) ويشيل التشكيل.

In [2]:
"""Text normalization shared by every approach."""
import re

_AR_DIGITS = str.maketrans("٠١٢٣٤٥٦٧٨٩٫٬", "0123456789.,")
_DIACRITICS = re.compile(r"[ً-ْـ]")  # harakat + tatweel


def to_western_digits(text: str) -> str:
    return text.translate(_AR_DIGITS)


def normalize_letters(text: str) -> str:
    """Unify Arabic letter variants so keyword and word-number matching is robust."""
    text = _DIACRITICS.sub("", text)
    text = re.sub("[إأآ]", "ا", text)
    text = text.replace("ى", "ي").replace("ة", "ه")
    return text


def normalize(text: str) -> str:
    return normalize_letters(to_western_digits(text)).lower()


## 2) استخراج المبلغ — `amounts.py` (قواعد، مش موديل)
- `NUM_RE` بيلاقي الأرقام، و`extract_amount` بيدّي نقط لكل رقم: جنبه "ب" أو "جنيه" (+)، جنبه "رصيد" أو وحدة زي "كيلو" (−).
- `_word_value` بيحوّل الأرقام بالكلام: ميتين، ألف وخمسمية، باكو (1000)، أرنب (مليون).

In [3]:
"""Rule-based amount extraction: digits, Arabic digits, "9.5k" and Egyptian number words.

Numbers are deterministic, so they are parsed with rules instead of a learned model.
The learned models decide *which* span is the amount (spaCy NER); this module turns
any span into a number.
"""
import re


UNITS = {
    "واحد": 1, "واحده": 1, "اتنين": 2, "اثنين": 2, "تلاته": 3, "تلات": 3, "تلت": 3, "ثلاثه": 3,
    "اربعه": 4, "اربع": 4, "خمسه": 5, "خمس": 5, "سته": 6, "ست": 6, "سبعه": 7, "سبع": 7,
    "تمانيه": 8, "تمان": 8, "تمن": 8, "تسعه": 9, "تسع": 9, "عشره": 10, "عشر": 10,
    "حداشر": 11, "اتناشر": 12, "تلتاشر": 13, "اربعتاشر": 14, "اربعطاشر": 14, "خمستاشر": 15,
    "خمسطاشر": 15, "ستاشر": 16, "سبعتاشر": 17, "سبعطاشر": 17, "تمنتاشر": 18, "تمانتاشر": 18,
    "تسعتاشر": 19, "تسعطاشر": 19,
    "عشرين": 20, "تلاتين": 30, "اربعين": 40, "خمسين": 50, "ستين": 60, "سبعين": 70,
    "تمانين": 80, "تسعين": 90,
}
HUNDREDS = {
    "ميه": 100, "ميت": 100, "مايه": 100, "ميتين": 200, "تلتميه": 300, "ربعميه": 400,
    "خمسميه": 500, "ستميه": 600, "سبعميه": 700, "تمنميه": 800, "تسعميه": 900,
    "متين": 200, "اربعميه": 400, "ثلاثميه": 300, "ثمانميه": 800, "تمانميه": 800,
}
THOUSANDS = {"الف", "الاف", "تلاف", "الالاف", "باكو"}  # باكو: Egyptian slang for 1,000
MILLIONS = {"مليون", "ملايين", "ارنب"}  # ارنب: slang for 1,000,000
HALF, QUARTER = {"نص", "نصف"}, {"ربع"}
VOCAB = set(UNITS) | set(HUNDREDS) | THOUSANDS | MILLIONS | HALF | QUARTER | {"الفين"}

CURRENCY_RE = r"(?:جنيه|جنيها|جنية|جنيهات|ج\.م|ج|egp|le|l\.e|pound|pounds|دولار|usd|\$)"
USD_RE = re.compile(r"\$|usd|دولار|dollar", re.I)
NUM_RE = re.compile(r"(?<![\d*/:.,a-z])(\d{1,3}(?:,\d{3})+|\d+)(\.\d+)?(\s?k\b|\s?(?:الف|ألف|باكو))?(?![\d/:a-jl-z]|\.\d|%)", re.I)
HALF_THOUSAND_RE = re.compile(r"(?<![\d.])(\d+)\s?(?:و\s?)?نص\s?(?:الف|ألف|باكو)")
BEFORE_CUE = re.compile(r"(?:\bب\s?ـ?\s?|بمبلغ\s?|مبلغ\s?|\bfor\s|\bpaid\s|\bspent\s|" + CURRENCY_RE + r"\s?)$", re.I)
AFTER_CUE = re.compile(r"^\s?" + CURRENCY_RE + r"(?![a-z؀-ۿ])", re.I)
QUANTITY_AFTER = re.compile(r"^\s?(?:كيلو|علب|علبه|شهور|شهر|ايام|يوم|قطع|قطعه|حته|حتت|تيشيرت|تيشيرتات|مرات|مره|نفر|ساعات|ساعه|x\b|pcs|kg|months?|days?|items?)", re.I)
BALANCE_BEFORE = re.compile(r"(?:\bbal\b\.?|balance|رصيد|متاح|المتبقي|\bref\b\.?|رقم|otp|كود)[^\d]{0,22}$", re.I)


def _word_value(tokens):
    """Value of a run of Egyptian number words, e.g. ['ألف', 'وخمسمية'] -> 1500."""
    total, current, last_mult, seen = 0.0, 0.0, 1, False
    for raw in tokens:
        w = _strip_prefix(raw)
        if w in HALF or w in QUARTER:
            frac = 0.5 if w in HALF else 0.25
            if last_mult >= 1000:
                total += last_mult * frac
            elif last_mult == 100:
                current += 100 * frac
            else:
                current += frac
        elif w in UNITS:
            current += UNITS[w]
        elif w in HUNDREDS:
            current = current * 100 if w in ("ميه", "ميت", "مايه") and 0 < current < 10 else current + HUNDREDS[w]
            last_mult = 100
        elif w == "الفين":
            total += 2000
            last_mult = 1000
        elif w in THOUSANDS:
            total += (current or 1) * 1000
            current, last_mult = 0, 1000
        elif w in MILLIONS:
            total += (current or 1) * 1_000_000
            current, last_mult = 0, 1_000_000
        else:
            continue
        seen = True
    return total + current if seen else None


def _strip_prefix(token: str) -> str:
    w = normalize_letters(token).strip("،,.!?؟").replace("مائه", "ميه")
    for prefix in ("وب", "و", "ب"):
        if w.startswith(prefix) and w[len(prefix):] in VOCAB:
            return w[len(prefix):]
    return w


def parse_number_text(span: str):
    """Convert an amount span ('1,200', '١٢٠٠', '9.5k', 'ألف وخمسمية') to a float, or None."""
    t = to_western_digits(span).strip().lower()
    m = re.search(r"(\d{1,3}(?:,\d{3})+|\d+)(\.\d+)?", t)
    if m:
        value = float(m.group(1).replace(",", "") + (m.group(2) or ""))
        rest = t[m.end():].strip()
        if re.match(r"(?:و\s?)?نص\s?(?:الف|ألف|باكو)", rest):
            return (value + 0.5) * 1000
        if rest.startswith(("k", "الف", "ألف", "باكو")):
            value *= 1000
        return value
    return _word_value(t.split())


def _word_runs(text: str):
    """Yield (start, end, value, n_tokens, has_multiplier) for runs of number words."""
    tokens = list(re.finditer(r"\S+", text))
    i = 0
    while i < len(tokens):
        if _strip_prefix(tokens[i].group()) in VOCAB:
            j = i
            while j + 1 < len(tokens) and (_strip_prefix(tokens[j + 1].group()) in VOCAB or tokens[j + 1].group() == "و"):
                j += 1
            run = [tok.group() for tok in tokens[i:j + 1]]
            value = _word_value(run)
            if value:
                words = {_strip_prefix(t) for t in run}
                has_mult = bool(words & (set(HUNDREDS) | THOUSANDS | MILLIONS | {"الفين"}))
                yield tokens[i].start(), tokens[j].end(), value, len(run), has_mult
            i = j + 1
        else:
            i += 1


def extract_amount(text: str):
    """Return {'amount', 'currency', 'span'} using rules only. amount is None when nothing is found."""
    t = to_western_digits(text).lower()  # same length as text, so offsets stay valid
    candidates = []
    for m in HALF_THOUSAND_RE.finditer(t):  # "2 ونص ألف" = 2500
        candidates.append((4, (float(m.group(1)) + 0.5) * 1000, m.start(), m.end()))
    for m in NUM_RE.finditer(t):
        value = float(m.group(1).replace(",", "") + (m.group(2) or ""))
        if m.group(3):
            value *= 1000
        before, after = t[max(0, m.start() - 30):m.start()], t[m.end():m.end() + 14]
        score = 0
        if BEFORE_CUE.search(before[-12:]):
            score += 3
        if AFTER_CUE.search(after):
            score += 3
        if m.group(3):
            score += 1
        if QUANTITY_AFTER.search(after):
            score -= 3
        if BALANCE_BEFORE.search(before):
            score -= 5
        if value == 0:
            continue
        candidates.append((score, value, m.start(), m.end()))
    if not candidates:
        for start, end, value, n_tokens, has_mult in _word_runs(text):
            near_currency = bool(AFTER_CUE.search(t[end:end + 14]))
            if n_tokens == 1 and not has_mult and not near_currency and value < 11:
                continue  # "واحد صاحبي" is not an amount
            candidates.append((2 if near_currency else 0, value, start, end))
    if not candidates:
        return {"amount": None, "currency": None, "span": None}
    score, value, start, end = max(candidates, key=lambda c: (c[0], c[1]))
    currency = "USD" if USD_RE.search(t) else "EGP"
    return {"amount": value, "currency": currency, "span": (start, end)}


## 3) الداتا — `data.py`
تحميل الـ 2000 رسالة، والتقسيم الثابت 80/20 (stratified حسب النية)، وتحويل الكيانات لمواضع حروف.

In [4]:
"""Dataset loading, the fixed train/test split, and label mapping."""
import glob
import json
import os
import random
from collections import defaultdict

ROOT = os.path.abspath(".")  # the notebook runs from the nlp_prototype folder
DATA = os.path.join(ROOT, "data")
MODELS = os.path.join(ROOT, "models")

INTENTS = ["log_expense", "log_income", "log_savings", "log_transfer", "question", "greeting", "other"]
TX_INTENTS = {"log_expense", "log_income", "log_savings", "log_transfer"}
# Coarse type used to compare against the hand-written set, which only has a "type" column.
COARSE = {"log_expense": "expense", "log_income": "income", "log_savings": "savings",
          "log_transfer": "transfer", "question": "none", "greeting": "none", "other": "none"}


def load_jsonl(path):
    with open(path, encoding="utf-8") as fh:
        return [json.loads(line) for line in fh if line.strip()]


def load_synthetic():
    recs = []
    for path in sorted(glob.glob(os.path.join(DATA, "train", "*.jsonl"))):
        recs.extend(load_jsonl(path))
    return recs


def split(recs, test_ratio=0.2, seed=42):
    """Deterministic split, stratified by intent so every intent appears in both parts."""
    by_intent = defaultdict(list)
    for r in recs:
        by_intent[r["intent"]].append(r)
    rng = random.Random(seed)
    train, test = [], []
    for intent in sorted(by_intent):
        group = sorted(by_intent[intent], key=lambda r: r["id"])
        rng.shuffle(group)
        n_test = max(1, round(len(group) * test_ratio))
        test.extend(group[:n_test])
        train.extend(group[n_test:])
    return train, test


def load_handwritten():
    """80 hand-written messages (different author and style from the synthetic data)."""
    recs = load_jsonl(os.path.join(DATA, "eval_handwritten.jsonl"))
    for r in recs:
        r["coarse"] = r["type"]
    return recs


def entity_offsets(rec):
    """Convert {'label', 'text'} entities to (start, end, label) character offsets."""
    out = []
    for e in rec["entities"]:
        start = rec["text"].find(e["text"])
        if start >= 0:
            out.append((start, start + len(e["text"]), e["label"]))
    return out


## 4) نوع الحاجة — `item_types.py`
قاموس كلمات: لاب ← إلكترونيات، كوتشي ← أحذية… و`rule_item` بيلقط الحاجة اللي بعد "اشتريت/جبت/طلبت" لو الـ NER فوّتها.

In [5]:
"""Item type (sub-category) lookup, e.g. "لاب" -> Electronics, "كوتشي" -> Shoes.

The training data has no item-type labels, so this is a keyword lexicon for now.
Once real data is labeled with item types, it can be replaced by a classifier.
"""
import re


# type: (Arabic label, keywords in normalized form: أ/إ/آ -> ا, ة -> ه, ى -> ي)
ITEM_TYPES = {
    "Electronics": ("إلكترونيات", "لاب لابتوب laptop موبايل تليفون فون ايفون iphone سامسونج samsung شاومي سماعه سماعات ايربودز airpods "
                    "شاحن تابلت ايباد ipad شاشه تلفزيون تليفزيون كيبورد ماوس بلايستيشن playstation ps5 كاميرا ساعه سمارت باور بانك"),
    "Clothes": ("ملابس", "تيشيرت تيشرت قميص بنطلون جينز جاكيت بلوزه فستان هدوم لبس عبايه شورت هودي hoodie بدله جيبه شراب بيجامه"),
    "Shoes": ("أحذية", "كوتشي كوتشيات جزمه صندل شبشب sneakers shoes"),
    "Groceries": ("بقالة", "عيش لبن جبنه خضار فاكهه رز سكر زيت لحمه فراخ بيض مكرونه تموين بقاله سمك groceries"),
    "Meals": ("أكل جاهز", "بيتزا برجر كشري شاورما فول طعميه سندوتش سندوتشات وجبه كريب فطير pizza burger koshary"),
    "Drinks": ("مشروبات وقهوة", "قهوه شاي لاتيه نسكافيه عصير كابتشينو coffee latte"),
    "Fuel": ("بنزين وشحن", "بنزين سولار بنزينه شحن العربيه fuel"),
    "Rides": ("مشاوير", "اوبر uber كريم careem تاكسي ميكروباص مترو اتوبيس توكتوك مشوار indrive"),
    "Medicine": ("أدوية", "دوا ادويه علاج فيتامين روشته"),
    "Medical": ("كشف وتحاليل", "كشف تحاليل اشعه دكتور اسنان"),
    "Books & courses": ("كتب وكورسات", "كتاب كتب كورس كورسات ملزمه دروس"),
    "Appliances": ("أجهزة منزلية", "تلاجه غساله بوتاجاز مكيف تكييف ميكروويف خلاط مكنسه سخان"),
    "Furniture": ("أثاث", "كنبه سرير ترابيزه دولاب كرسي مكتب"),
    "Personal care": ("عناية شخصية", "برفان ميكب شامبو كريم حلاق كوافير"),
    "Gifts": ("هدايا", "هديه هدايا ورد بوكيه"),
    "Mobile credit": ("رصيد وباقات", "رصيد باقه كارت فكه"),
    "Bills": ("فواتير", "فاتوره كهربا كهرباء مياه ميه غاز انترنت نت"),
}

_PREFIXES = ("وال", "بال", "لل", "ال", "و", "ب")
_LOOKUP = {kw: t for t, (_, words) in ITEM_TYPES.items() for kw in words.split() if " " not in kw}
_PHRASES = {"شحن العربيه": "Fuel", "باور بانك": "Electronics", "ساعه سمارت": "Electronics"}


def _variants(token):
    yield token
    for p in _PREFIXES:
        if token.startswith(p) and len(token) - len(p) >= 2:
            yield token[len(p):]


def item_type(item, text):
    """Return (type, Arabic label) from the item, falling back to the whole message; (None, None) if unknown."""
    for source in (item, text):
        if not source:
            continue
        t = normalize(source)
        for phrase, typ in _PHRASES.items():
            if phrase in t:
                return typ, ITEM_TYPES[typ][0]
        for token in re.findall(r"[\w]+", t):
            for v in _variants(token):
                if v in _LOOKUP:
                    typ = _LOOKUP[v]
                    return typ, ITEM_TYPES[typ][0]
    return None, None


# "اشتريت لاب من ..." -> "لاب": what follows a purchase verb, up to a place, amount or date word.
ITEM_RE = re.compile(
    r"(?:اشتريت|اشترينا|جبت|طلبت|خدت|bought|got|ordered)\s+(.+?)"
    r"(?=\s+(?:من|عند|في|ب|بـ|بمبلغ|ب?\d|امبارح|النهارده|النهاردة|from|at|for|on)\b|\s*[\d٠-٩,.،!؟?]|\s*$)",
    re.I)


def rule_item(text):
    m = ITEM_RE.search(text)
    if not m:
        return None
    name = m.group(1).strip()
    return name if 0 < len(name.split()) <= 4 else None


## 5) الطريقة A: القواعد (baseline) — `rules.py`
كلمات مفتاحية للنية والفئة. من غير تدريب. موجودة للمقارنة بس.

In [6]:
"""Approach A: keyword rules + rule-based amounts. No training. This is the baseline."""
import re


INTENT_KEYWORDS = [
    ("log_income", r"قبضت|مرتب|راتب|بونص|نزل(?:ي|لي)? في حسابي|استلمت|اتحولي|بعتلي|رجعلي|رجعتلي|استرجعت|refund|salary|bonus|received|got paid|ead+ya|عيديه|2abadt|2bdt|mortab"),
    ("log_savings", r"حوشت|حطيت .*(?:توفير|شهاده|حساب)|شهاده|جمعيه|ادخار|توفير|savings|saved|7awesht|gam3eya"),
    ("log_transfer", r"سلفت|حولت|تحويل|سحبت .*(?:atm|ماكينه)|سحب|lent|transfer|sallaft|7awelt"),
    ("log_expense", r"دفعت|اشتريت|جبت|صرفت|اشتركت|جددت|شحنت|طلبت|ركبت|سددت|خلصت|paid|bought|spent|dafa3t|eshtaret|gebt|saraft|اتغديت|فطرت|اتعشيت"),
]
QUESTION = re.compile(r"[?؟]|كام|ازاي|ليه|امتي|فين|ايه|هل |how|what|why|when")
GREETING = re.compile(r"^(?:اهلا|السلام|صباح|مساء|هاي|hi|hello|hey|ازيك|عامل ايه|شكرا|thanks)")

CATEGORY_KEYWORDS = [
    ("Food", r"اكل|غدا|عشا|فطار|فطرت|قهوه|كشري|فول|طعميه|بيتزا|برجر|شاورما|طلبات|talabat|عيش|خضار|فاكهه|سوبر ?ماركت|كارفور|food|lunch|dinner|coffee|grocer|koshary|عصير|شاي|حلويات"),
    ("Transport", r"اوبر|uber|كريم|careem|تاكسي|مترو|ميكروباص|بنزين|مواصلات|indrive|ديدي|didi|بنزينه|عربيه|قطر|اتوبيس|metro|taxi|fuel|bus|gas station"),
    ("Housing", r"ايجار|شقه|بواب|سباك|كهربائي|نقاش|rent|plumber"),
    ("Utilities", r"كهربا|كهرباء|ميه|مياه|غاز|نت|انترنت|فاتوره|رصيد|باقه|electric|water|internet|bill|recharge"),
    ("Subscriptions", r"اشتراك|نتفليكس|netflix|spotify|سبوتيفاي|شاهد|subscription|youtube premium|chatgpt"),
    ("Entertainment", r"سينما|خروجه|خرجت|ترفيه|ملاهي|بلايستيشن|cinema|movie|game|escape room|حفله"),
    ("Shopping", r"هدوم|لبس|كوتشي|جزمه|تيشيرت|بنطلون|موبايل|سماعه|لابتوب|هديه|امازون|amazon|نون|noon|جوميا|jumia|مول|shoes|clothes|gift"),
    ("Health", r"دكتور|كشف|دوا|صيدليه|تحاليل|جيم|gym|doctor|pharmacy|medicine|اسنان"),
    ("Education", r"كورس|كتب|كتاب|جامعه|مدرسه|دروس|مصاريف الدراسه|course|tuition|books|school"),
    ("Debt", r"قسط|سددت|دين|رجعت .*مستلف|installment|loan"),
    ("Salary", r"مرتب|راتب|بونص|salary|bonus|mortab"),
    ("Freelance", r"فريلانس|freelance|مشروع|عميل|client|خمسات|upwork"),
    ("Gift", r"عيديه|هديه|ادتني|ادالي|gift"),
    ("Refund", r"مرتجع|refund|رجعلي|استرجعت|استرداد"),
    ("Savings", r"توفير|شهاده|جمعيه|ادخار|savings"),
    ("Lending", r"سلفت|lent"),
    ("Transfer", r"حولت|تحويل|سحب|transfer|atm"),
]
ALLOWED = {
    "log_expense": {"Food", "Transport", "Housing", "Utilities", "Subscriptions", "Entertainment", "Shopping", "Health", "Education", "Debt", "Other"},
    "log_income": {"Salary", "Freelance", "Gift", "Refund", "Other"},
    "log_savings": {"Savings"},
    "log_transfer": {"Transfer", "Lending"},
}


class RulesApproach:
    name = "A. Rules (keywords + regex)"

    def train(self, recs):
        pass

    def size_bytes(self):
        return 0

    def predict(self, text):
        t = normalize(text)
        amount = extract_amount(text)
        intent = next((name for name, pat in INTENT_KEYWORDS if re.search(pat, t)), None)
        if intent is None:
            if GREETING.search(t):
                intent = "greeting"
            elif amount["amount"] is not None:
                intent = "log_expense"  # "netflix 240": a bare amount is most likely a purchase
            elif QUESTION.search(t):
                intent = "question"
            else:
                intent = "other"
        category = None
        if intent in ALLOWED:
            category = next((c for c, pat in CATEGORY_KEYWORDS if c in ALLOWED[intent] and re.search(pat, t)), None)
            if category is None:
                category = {"log_savings": "Savings", "log_transfer": "Transfer"}.get(intent, "Other")
        return {"intent": intent, "category": category, "category_conf": None, **amount, "entities": []}


## 6) الطريقة B: TF-IDF + Logistic Regression — `tfidf.py`
- `TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5))`: بيقطّع النص لقطع حروف من 2 لـ 5.
- `LogisticRegression`: بيتعلم وزن لكل قطعة. **متدرب من الصفر** على داتتنا.
- موديلين: واحد للنية، وواحد للفئة (الفئة مسبوقة بالنية عشان "Other" بتاعة الدخل غير بتاعة المصروف).

In [7]:
"""Approach B: character n-gram TF-IDF + logistic regression (intent and category) + rule-based amounts."""
import os
import pickle

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline



def _model():
    # Character n-grams cope with Arabic prefixes, typos and Franco-Arabic without a tokenizer.
    return make_pipeline(
        TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True, min_df=1),
        LogisticRegression(max_iter=3000, C=10),
    )


class TfidfApproach:
    name = "B. TF-IDF + Logistic Regression"
    path = os.path.join(MODELS, "tfidf.pkl")

    def train(self, recs):
        self.intent = _model().fit([normalize(r["text"]) for r in recs], [r["intent"] for r in recs])
        tx = [r for r in recs if r["intent"] in TX_INTENTS and r["category"]]
        # Prefix the intent so "Other" income and "Other" expense stay distinct.
        self.category = _model().fit([normalize(r["text"]) for r in tx], [f"{r['intent']}|{r['category']}" for r in tx])
        os.makedirs(MODELS, exist_ok=True)
        with open(self.path, "wb") as fh:
            pickle.dump((self.intent, self.category), fh)

    def load(self):
        with open(self.path, "rb") as fh:
            self.intent, self.category = pickle.load(fh)
        return self

    def size_bytes(self):
        return os.path.getsize(self.path)

    def predict(self, text):
        t = normalize(text)
        intent = self.intent.predict([t])[0]
        category, conf = None, None
        if intent in TX_INTENTS:
            probs = self.category.predict_proba([t])[0]
            classes = self.category.classes_
            # Only categories valid for the predicted intent.
            allowed = [(p, c) for p, c in zip(probs, classes) if c.startswith(intent + "|")]
            if allowed:
                p, c = max(allowed)
                category, conf = c.split("|", 1)[1], p / sum(p2 for p2, _ in allowed)
        return {"intent": intent, "category": category, "category_conf": conf, **extract_amount(text), "entities": []}


## 7) الطريقة C: spaCy — `spacy_model.py`
- `spacy.blank("ar")`: موديل **فاضي** (تقطيع كلام بس)، وبنزوّد قاعدة تفصل "ب1200" لـ "ب" + "1200".
- بنضيف `ner` (الحاجة، والمحل، والتاريخ، والمبلغ…) و`textcat` (النية)، ونموذج تاني للفئة، وكلهم **متدربين من الصفر** بـ `nlp.update` لمدة 15 epoch.

In [8]:
"""Approach C: spaCy pipelines (NER + intent textcat, and a category textcat), CPU-only.

The NER finds which span is the amount, the item, the merchant, etc. The span is then turned
into a number by amounts.parse_number_text, with the regex extractor as a fallback.
"""
import os
import random
import shutil

import spacy
from spacy.training import Example
from spacy.util import compile_infix_regex, fix_random_seed, minibatch


AR = r"ء-يـ"
DIGIT = r"0-9٠-٩"


def make_nlp():
    nlp = spacy.blank("ar")
    # Split "ب1200" into "ب" + "1200" so the amount can be its own entity.
    infixes = list(nlp.Defaults.infixes) + [rf"(?<=[{AR}])(?=[{DIGIT}])", rf"(?<=[{DIGIT}])(?=[{AR}])"]
    nlp.tokenizer.infix_finditer = compile_infix_regex(infixes).finditer
    return nlp


def _train(nlp, examples, epochs, seed=0):
    fix_random_seed(seed)
    optimizer = nlp.initialize(lambda: examples)
    rng = random.Random(seed)
    for _ in range(epochs):
        rng.shuffle(examples)
        losses = {}
        for batch in minibatch(examples, size=16):
            nlp.update(batch, sgd=optimizer, drop=0.2, losses=losses)
    return nlp


class SpacyApproach:
    name = "C. spaCy (NER + textcat)"
    main_dir = os.path.join(MODELS, "spacy_main")
    cat_dir = os.path.join(MODELS, "spacy_category")

    def __init__(self, epochs=15):
        self.epochs = epochs
        self.misaligned = 0

    def train(self, recs):
        main = make_nlp()
        main.add_pipe("ner")
        main.add_pipe("textcat")  # exclusive: exactly one intent per message
        examples = []
        for r in recs:
            doc = main.make_doc(r["text"])
            ents = []
            for start, end, label in entity_offsets(r):
                span = doc.char_span(start, end, label=label, alignment_mode="strict")
                if span is None:
                    self.misaligned += 1  # span boundary falls inside a token; skip that entity
                else:
                    ents.append((start, end, label))
            cats = {i: float(i == r["intent"]) for i in sorted({x["intent"] for x in recs})}
            examples.append(Example.from_dict(doc, {"entities": ents, "cats": cats}))
        self.main = _train(main, examples, self.epochs)

        cat_nlp = make_nlp()
        cat_nlp.add_pipe("textcat")
        tx = [r for r in recs if r["intent"] in TX_INTENTS and r["category"]]
        labels = sorted({f"{r['intent']}|{r['category']}" for r in tx})
        cat_examples = [Example.from_dict(cat_nlp.make_doc(r["text"]),
                                          {"cats": {l: float(l == f"{r['intent']}|{r['category']}") for l in labels}})
                        for r in tx]
        self.cat = _train(cat_nlp, cat_examples, self.epochs)

        for d, nlp in ((self.main_dir, self.main), (self.cat_dir, self.cat)):
            shutil.rmtree(d, ignore_errors=True)
            nlp.to_disk(d)

    def load(self):
        self.main, self.cat = spacy.load(self.main_dir), spacy.load(self.cat_dir)
        return self

    def size_bytes(self):
        return sum(os.path.getsize(os.path.join(dp, f)) for d in (self.main_dir, self.cat_dir)
                   for dp, _, files in os.walk(d) for f in files)

    def predict(self, text):
        doc = self.main(text)
        intent = max(doc.cats, key=doc.cats.get)
        entities = [{"label": e.label_, "text": e.text} for e in doc.ents]
        result = {"intent": intent, "category": None, "category_conf": None,
                  "amount": None, "currency": None, "span": None, "entities": entities}
        amount_ents = [e for e in doc.ents if e.label_ == "AMOUNT"]
        value = parse_number_text(amount_ents[0].text) if amount_ents else None
        if value:
            result.update(amount=value, currency="USD" if USD_RE.search(to_western_digits(text).lower()) else "EGP",
                          span=(amount_ents[0].start_char, amount_ents[0].end_char))
        else:
            result.update(extract_amount(text))  # fallback when NER finds no usable span
        if intent in TX_INTENTS:
            cats = self.cat(text).cats
            allowed = {k: v for k, v in cats.items() if k.startswith(intent + "|")}
            if allowed:
                best = max(allowed, key=allowed.get)
                result["category"] = best.split("|", 1)[1]
                result["category_conf"] = allowed[best] / sum(allowed.values())
        return result


## 8) الـ pipeline النهائي — `pipeline.py`
المبلغ بالقواعد ← النوع والفئة بـ TF-IDF (لو الثقة أقل من 25% تبقى "أخرى") ← الكيانات بـ spaCy + قواعد احتياطية للمحل والحاجة ← نوع الحاجة من القاموس ← رد جاهز بالمصري.

In [9]:
"""The recommended pipeline: what the chatbot would actually run.

- amount and currency: rules (amounts.py), because numbers are deterministic
- intent and category: TF-IDF + logistic regression (best classifier in the comparison)
- item, merchant, date, person: spaCy NER, plus a rule for "من محل ..." when NER misses the merchant
- reply: fixed Egyptian-Arabic templates, so the bot never invents text

The output matches the ERD: `extracted_data` is stored in chat_messages.extracted_data and
`status` maps to chat_messages.extraction_status. A complete message is saved directly (`confirmed`);
the bot picks the category itself and only asks when the amount is missing.
"""
import re


# Below this confidence the guess is usually wrong, so "Other" is safer (tested on both test sets).
OTHER_FALLBACK_BELOW = 0.25

# "من محل ابو جزم", "عند الجزار", "from Zara", "at Costa": the place ends before the amount, a date word or punctuation.
MERCHANT_RE = re.compile(
    r"(?:\bمن عند|\bمن|\bعند|\bfrom|\bat)\s+(?:محل\s+|shop\s+)?"
    r"(.+?)(?=\s+(?:ب\s?ـ?\s?\d|بمبلغ|ب\s|\d|مرتجع|امبارح|النهارده|النهاردة|الصبح|بالليل|يوم|for\b|yesterday|today)|\s*[\d٠-٩,.،!؟?]|\s*$)",
    re.I)
NOT_A_MERCHANT = {"البيت", "الشغل", "حسابي", "الحساب", "المرتب", "ماما", "بابا", "صاحبي", "home", "work"}

TYPE_AR = {"log_expense": "مصروف", "log_income": "دخل", "log_savings": "ادخار", "log_transfer": "تحويل"}
CATEGORY_AR = {
    "Food": "أكل", "Transport": "مواصلات", "Housing": "سكن", "Utilities": "فواتير", "Subscriptions": "اشتراكات",
    "Entertainment": "خروج وترفيه", "Shopping": "مشتريات", "Health": "صحة", "Education": "تعليم", "Debt": "أقساط وديون",
    "Other": "أخرى", "Salary": "مرتب", "Freelance": "فريلانس", "Gift": "هدية", "Refund": "مرتجع",
    "Savings": "ادخار", "Transfer": "تحويل", "Lending": "سلفة",
}


def _rule_merchant(text, item):
    m = MERCHANT_RE.search(text)
    if not m:
        return None
    name = m.group(1).strip()
    if not name or name in NOT_A_MERCHANT or name == item or len(name.split()) > 4:
        return None
    return name


def _money(amount, currency):
    value = f"{amount:,.0f}" if amount == int(amount) else f"{amount:,.2f}"
    return f"{value} {'دولار' if currency == 'USD' else 'جنيه'}"


class Assistant:
    def __init__(self):
        self.classifier = TfidfApproach().load()
        self.ner = SpacyApproach().load()

    def handle(self, text):
        cls = self.classifier.predict(text)
        intent = cls["intent"]
        if intent not in TX_INTENTS:
            replies = {
                "greeting": "أهلاً بيك 👋 ابعتلي أي مصروف أو دخل وأنا أسجلهولك. مثلاً: «دفعت 120 أكل»",
                "question": "الأسئلة عن مصاريفك جاية في النسخة الجاية 🙏 دلوقتي أقدر أسجلك أي مصروف أو دخل.",
            }
            return {"status": "not_applicable", "extracted_data": {"intent": intent},
                    "reply": replies.get(intent, "مش فاهم قصدك 🤔 اكتب المبلغ وصرفته على إيه، زي «جبت عيش ب 20».")}

        ents = {}
        for e in self.ner.main(text).ents:
            ents.setdefault(e.label_.lower(), e.text)
        amount = extract_amount(text)
        category, conf = cls["category"], cls["category_conf"]
        if category is None or (conf is not None and conf < OTHER_FALLBACK_BELOW and "Other" in ALLOWED[intent]):
            category = "Other" if "Other" in ALLOWED[intent] else sorted(ALLOWED[intent])[0]
        merchant, item = ents.get("merchant"), ents.get("item")
        if item is None and intent == "log_expense":
            item = rule_item(text)  # "اشتريت لاب من مول البستان" -> "لاب"
        if merchant is None and intent in ("log_expense", "log_income"):
            merchant = _rule_merchant(text, item)
        if merchant and item and merchant in item and merchant != item:
            # NER sometimes swallows the place into the item: "لحمة من عند الجزار" -> "لحمة"
            item = re.sub(r"\s*(?:من عند|من|عند|from|at)\s*$", "", item[:item.find(merchant)].strip()) or item
        data = {"intent": intent, "type": intent.removeprefix("log_"), "amount": amount["amount"],
                "currency": amount["currency"], "category": category,
                "category_confidence": round(conf, 3) if conf is not None else None,
                "item": item, "item_type": None, "item_type_ar": None, "merchant": merchant,
                "date": ents.get("date"), "person": ents.get("person")}
        if intent == "log_expense":
            data["item_type"], data["item_type_ar"] = item_type(item, text)

        return self._finish(data)

    def follow_up(self, draft, text=None, category=None):
        """Complete a `needs_info` draft with the user's next answer (an amount or a chosen category)."""
        data = dict(draft)
        if category is not None and category in ALLOWED.get(data["intent"], ()):
            data["category"], data["category_confidence"] = category, 1.0
        elif text is not None and "amount" in data.get("missing", []):
            amount = extract_amount(text)
            if amount["amount"] is None:
                return {"status": "needs_info", "extracted_data": data, "reply": "مش لاقي مبلغ 🤔 اكتب الرقم بس، زي «350»."}
            data["amount"], data["currency"] = amount["amount"], amount["currency"]
        else:
            return self.handle(text or "")
        return self._finish(data)

    def _finish(self, data):
        intent = data["intent"]
        what = data["item"] or data["merchant"] or "ده"
        if data["amount"] is None:
            data["missing"] = ["amount"]
            return {"status": "needs_info", "extracted_data": data, "reply": f"تمام، {what} كان بكام؟"}
        data["missing"] = []
        details = " ".join(x for x in (data["item"], f"من {data['merchant']}" if data["merchant"] else None, data["date"]) if x)
        reply = (f"اتسجل ✅ {TYPE_AR[intent]}: {_money(data['amount'], data['currency'])} – {CATEGORY_AR[data['category']]}"
                 f"{f' ({details})' if details else ''}")
        return {"status": "confirmed", "extracted_data": data, "reply": reply}


## 9) المقارنة — `evaluate.py`
بيدرّب الطرق الثلاثة على نفس الـ 80%، ويقيسهم على الـ 20% وعلى الـ 80 جملة اليدوي.

In [10]:
"""Train every approach on the same split and compare them.

Usage: python -m nlp_proto.evaluate
Writes results/results.md and results/results.json.
"""
import json
import os
import time



def _same_amount(pred, gold_amount, gold_currency):
    return pred["amount"] is not None and abs(pred["amount"] - gold_amount) < 1e-6 and pred["currency"] == gold_currency


def score(preds, gold, has_fine_intent):
    n = len(gold)
    m = {"n": n}
    coarse_pred = [COARSE[p["intent"]] for p in preds]
    coarse_gold = [COARSE[g["intent"]] if has_fine_intent else g["coarse"] for g in gold]
    m["type_acc"] = sum(a == b for a, b in zip(coarse_pred, coarse_gold)) / n
    if has_fine_intent:
        m["intent_acc"] = sum(p["intent"] == g["intent"] for p, g in zip(preds, gold)) / n
    with_amount = [(p, g) for p, g in zip(preds, gold) if g["amount"] is not None]
    m["amount_acc"] = sum(_same_amount(p, g["amount"], g["currency"]) for p, g in with_amount) / len(with_amount)
    with_cat = [(p, g) for p, g in zip(preds, gold) if g["category"] is not None]
    m["category_acc"] = sum(p["category"] == g["category"] for p, g in with_cat) / len(with_cat)
    ok = 0
    for p, g, cp, cg in zip(preds, gold, coarse_pred, coarse_gold):
        if cg == "none":
            ok += cp == "none"
        else:
            ok += (cp == cg and (g["amount"] is None or _same_amount(p, g["amount"], g["currency"]))
                   and (g["category"] is None or p["category"] == g["category"]))
    m["end_to_end"] = ok / n
    return m


def ner_scores(preds, gold):
    """Micro P/R/F1 over (label, text) pairs, excluding nothing."""
    tp = fp = fn = 0
    for p, g in zip(preds, gold):
        pred = {(e["label"], e["text"]) for e in p["entities"]}
        true = {(label, g["text"][s:e]) for s, e, label in entity_offsets(g)}
        tp += len(pred & true); fp += len(pred - true); fn += len(true - pred)
    prec = tp / (tp + fp) if tp + fp else 0
    rec = tp / (tp + fn) if tp + fn else 0
    return {"precision": prec, "recall": rec, "f1": 2 * prec * rec / (prec + rec) if prec + rec else 0}


def evaluate_all():
    synthetic = load_synthetic()
    train, test = split(synthetic)
    handwritten = load_handwritten()
    results = {"train_size": len(train), "test_size": len(test), "handwritten_size": len(handwritten), "approaches": []}
    for approach in (RulesApproach(), TfidfApproach(), SpacyApproach()):
        t0 = time.perf_counter()
        approach.train(train)
        train_s = time.perf_counter() - t0
        t0 = time.perf_counter()
        test_preds = [approach.predict(r["text"]) for r in test]
        hw_preds = [approach.predict(r["text"]) for r in handwritten]
        latency_ms = (time.perf_counter() - t0) / (len(test) + len(handwritten)) * 1000
        row = {"name": approach.name, "train_seconds": round(train_s, 1), "latency_ms": round(latency_ms, 2),
               "size_mb": round(approach.size_bytes() / 1e6, 2),
               "synthetic_test": score(test_preds, test, True), "handwritten": score(hw_preds, handwritten, False)}
        if isinstance(approach, SpacyApproach):
            row["ner"] = ner_scores(test_preds, test)
            row["misaligned_entities_skipped"] = approach.misaligned
        row["handwritten_errors"] = [
            {"id": g["id"], "text": g["text"], "gold": [g["coarse"], g["amount"], g["category"]],
             "pred": [COARSE[p["intent"]], p["amount"], p["category"]]}
            for p, g in zip(hw_preds, handwritten)
            if not (COARSE[p["intent"]] == g["coarse"] and (g["amount"] is None or _same_amount(p, g["amount"], g["currency"]))
                    and (g["category"] is None or p["category"] == g["category"]))]
        results["approaches"].append(row)
        print(f"done: {approach.name}")

    os.makedirs(os.path.join(ROOT, "results"), exist_ok=True)
    with open(os.path.join(ROOT, "results", "results.json"), "w", encoding="utf-8") as fh:
        json.dump(results, fh, ensure_ascii=False, indent=2)
    md = render(results)
    with open(os.path.join(ROOT, "results", "results.md"), "w", encoding="utf-8") as fh:
        fh.write(md)
    print(md)


def render(r):
    pct = lambda x: f"{x * 100:.1f}%"
    lines = [f"Train: {r['train_size']} synthetic · Test: {r['test_size']} synthetic (held out) + {r['handwritten_size']} hand-written", "",
             "| Approach | Set | Type | Intent | Amount | Category | End-to-end |", "|---|---|---|---|---|---|---|"]
    for a in r["approaches"]:
        s, h = a["synthetic_test"], a["handwritten"]
        lines.append(f"| {a['name']} | synthetic test | {pct(s['type_acc'])} | {pct(s['intent_acc'])} | {pct(s['amount_acc'])} | {pct(s['category_acc'])} | **{pct(s['end_to_end'])}** |")
        lines.append(f"| | hand-written | {pct(h['type_acc'])} | – | {pct(h['amount_acc'])} | {pct(h['category_acc'])} | **{pct(h['end_to_end'])}** |")
    lines += ["", "| Approach | Model size | Latency / message | Training time |", "|---|---|---|---|"]
    for a in r["approaches"]:
        lines.append(f"| {a['name']} | {a['size_mb']} MB | {a['latency_ms']} ms | {a['train_seconds']} s |")
    for a in r["approaches"]:
        if "ner" in a:
            n = a["ner"]
            lines += ["", f"spaCy NER on synthetic test (exact span match): precision {pct(n['precision'])}, recall {pct(n['recall'])}, F1 **{pct(n['f1'])}**. Entities skipped in training because of token misalignment: {a['misaligned_entities_skipped']}."]
    return "\n".join(lines) + "\n"


## 10) الـ Router — `router.py`
أول خطوة لأي رسالة: غرضها إيه؟ تسجيل معاملة، ولا سؤال عن مصاريفي، ولا توقع، ولا أسواق، ولا نصيحة، ولا مسح، ولا تحية، ولا خارج النطاق؟
موديل TF-IDF + Logistic Regression تاني متدرب على `data/router/`، ومعاه قاعدة صغيرة لصيغة المستقبل ("هصرف" ← توقع).

In [11]:
"""Message router: decides what a chat message is for, before any extraction happens.

Routes (matching the team diagram: Classifier / Forecast / APIs + LLM):
  transaction    -> transaction extraction pipeline (FR-010..FR-013)
  spending_query -> computed from the user's own records (FR-018)
  forecast       -> spending projection from the user's records
  market         -> economic/market data APIs + LLM explanation (FR-023)
  advice         -> LLM guidance with a minimal data summary (FR-022, FR-029)
  manage         -> undo/delete on recorded entries (FR-016)
  greeting, out_of_scope -> fixed replies

Usage: python -m nlp_proto.router   trains, evaluates on a held-out 20%, saves models/router.pkl
"""
import collections
import glob
import json
import os
import pickle
import random
import re

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline


ROUTES = ["transaction", "spending_query", "forecast", "market", "advice", "manage", "greeting", "out_of_scope"]
ROUTE_AR = {"transaction": "تسجيل معاملة", "spending_query": "سؤال عن مصاريفي", "forecast": "توقع",
            "market": "أسواق واقتصاد", "advice": "نصيحة", "manage": "تعديل/مسح", "greeting": "تحية",
            "out_of_scope": "خارج النطاق"}
MAX_TRANSACTION_EXAMPLES = 500  # keep the 1,700+ transaction messages from swamping the other routes
# Character n-grams see "هصرف" (will spend) and "صرفت" (spent) as near-identical, so a question in the
# Egyptian future tense that the model calls spending_query is a forecast.
FUTURE_RE = re.compile(
    r"(?:^|\s)(?:ه|هي|هن)(?:صرف|حوش|وفر|دفع|فضل|تبقي|كفي|خلص|قبض|يفضل|يتبقي|يكفي|يخلص)\w*"
    r"|\bwill i\b|\bam i going to\b|\bhow much will\b|\bha(?:sraf|7awesh|dfa3)\b|\bhayfdal\b", re.I)


def load_router_data(seed=42):
    """Combine new router data, relabelled questions/corrections, and a sample of transaction messages."""
    synthetic = load_synthetic()
    by_id = {r["id"]: r for r in synthetic}
    items, relabels = [], {}
    for path in sorted(glob.glob(os.path.join(DATA, "router", "*.jsonl"))):
        for r in load_jsonl(path):
            if "relabel_of" in r:
                relabels[r["relabel_of"]] = r["route"]
            else:
                items.append((r["text"], r["route"]))
    for rid, route in relabels.items():
        items.append((by_id[rid]["text"], route))
    items += [(r["text"], "greeting") for r in synthetic if r["intent"] == "greeting"]
    tx = [r["text"] for r in synthetic if r["intent"] in TX_INTENTS]
    random.Random(seed).shuffle(tx)
    items += [(t, "transaction") for t in tx[:MAX_TRANSACTION_EXAMPLES]]
    return items


def split_router(items, test_ratio=0.2, seed=42):
    groups = collections.defaultdict(list)
    for text, route in items:
        groups[route].append((text, route))
    rng = random.Random(seed)
    train, test = [], []
    for route in sorted(groups):
        g = sorted(groups[route])
        rng.shuffle(g)
        n = max(1, round(len(g) * test_ratio))
        test += g[:n]
        train += g[n:]
    return train, test


class Router:
    path = os.path.join(MODELS, "router.pkl")
    unsure_below = 0.35  # below this the router asks what the user meant

    def train(self, items, save=True):
        self.model = make_pipeline(
            TfidfVectorizer(analyzer="char_wb", ngram_range=(2, 5), sublinear_tf=True),
            LogisticRegression(max_iter=3000, C=10, class_weight="balanced"),
        ).fit([normalize(t) for t, _ in items], [r for _, r in items])
        if save:
            os.makedirs(MODELS, exist_ok=True)
            with open(self.path, "wb") as fh:
                pickle.dump(self.model, fh)
        return self

    def load(self):
        with open(self.path, "rb") as fh:
            self.model = pickle.load(fh)
        return self

    def predict(self, text):
        probs = self.model.predict_proba([normalize(text)])[0]
        ranked = sorted(zip(probs, self.model.classes_), reverse=True)
        (p, route), (p2, route2) = ranked[0], ranked[1]
        route = str(route)
        if route == "spending_query" and FUTURE_RE.search(normalize(text)):
            route = "forecast"
        return {"route": route, "confidence": float(p), "runner_up": str(route2), "unsure": bool(p < self.unsure_below)}


def train_router():
    items = load_router_data()
    train, test = split_router(items)
    router = Router().train(train, save=False)
    preds = [router.predict(t)["route"] for t, _ in test]
    gold = [r for _, r in test]
    acc = sum(p == g for p, g in zip(preds, gold)) / len(gold)
    per_route = {r: (sum(p == g == r for p, g in zip(preds, gold)), gold.count(r)) for r in ROUTES if gold.count(r)}
    confusions = collections.Counter((g, p) for p, g in zip(preds, gold) if p != g).most_common(10)
    errors = [(t, g, p) for (t, g), p in zip(test, preds) if p != g][:25]
    lines = [f"Router: trained on {len(train)}, tested on {len(test)} held-out messages. Accuracy **{acc:.1%}**", "",
             "| Route | Correct / total | Accuracy |", "|---|---|---|"]
    lines += [f"| {r} ({ROUTE_AR[r]}) | {c} / {n} | {c / n:.1%} |" for r, (c, n) in per_route.items()]
    lines += ["", "Most common confusions (gold → predicted): " + ", ".join(f"{g} → {p} ({n})" for (g, p), n in confusions), "",
              "Sample errors:", ""] + [f"- `{t}`: gold {g}, predicted {p}" for t, g, p in errors]
    md = "\n".join(lines) + "\n"
    os.makedirs(os.path.join(ROOT, "results"), exist_ok=True)
    with open(os.path.join(ROOT, "results", "router_results.md"), "w", encoding="utf-8") as fh:
        fh.write(md)
    print(md)
    Router().train(items)  # final model on all data, used by the chat
    print(f"saved {Router.path} (trained on all {len(items)} messages)")


## 11) الـ LLM (اختياري) — `llm.py`
نداء لـ Gemini API بمكتبات بايثون الأساسية بس. بيشتغل لو `GEMINI_API_KEY` متحدد، وغير كده الكود بيرجع للردود بالقواعد.

In [12]:
"""Optional LLM call (Google Gemini REST API, standard library only).

Set GEMINI_API_KEY (and optionally GEMINI_MODEL) to enable it. Without a key, callers get None and
fall back to a fixed reply, so the prototype runs fully offline.
"""
import json
import os
import urllib.error
import urllib.request

DEFAULT_MODEL = "gemini-2.5-flash"


def enabled():
    return bool(os.environ.get("GEMINI_API_KEY"))


def generate(prompt, system, timeout=20):
    """Return the model's text, or None if the LLM is not configured or the call fails."""
    key = os.environ.get("GEMINI_API_KEY")
    if not key:
        return None
    model = os.environ.get("GEMINI_MODEL", DEFAULT_MODEL)
    body = json.dumps({
        "systemInstruction": {"parts": [{"text": system}]},
        "contents": [{"role": "user", "parts": [{"text": prompt}]}],
        "generationConfig": {"temperature": 0.4, "maxOutputTokens": 600},
    }).encode("utf-8")
    req = urllib.request.Request(
        f"https://generativelanguage.googleapis.com/v1beta/models/{model}:generateContent",
        data=body, headers={"Content-Type": "application/json", "x-goog-api-key": key}, method="POST")
    try:
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        return "".join(p.get("text", "") for p in data["candidates"][0]["content"]["parts"]).strip() or None
    except (urllib.error.URLError, KeyError, IndexError, ValueError, TimeoutError):
        return None


## 12) الـ Handlers — `handlers.py`
كل route ليه handler: حساب المصاريف من بيانات المستخدم، والتوقع (محتاج 5 معاملات على الأقل)، وأسعار العملات من API حقيقي ومعاه fallback، والنصيحة (الـ LLM بيشوف ملخص الأرقام بس)، والمسح.

In [13]:
"""Handlers for the non-transaction routes. Each returns {"reply", "data", optional "action"}.

`ledger` is the user's recorded transactions: [{"type", "amount", "currency", "category", "label", "date"}].
In the prototype the browser sends it with every message; in the real app it comes from the
database, scoped to the authenticated user only (FR-026, FR-029).
"""
import calendar
import datetime as dt
import json
import re
import time
import urllib.request


DISCLAIMER = "ℹ️ دي معلومات إرشادية مش استشارة مالية مرخصة."
MIN_TRANSACTIONS = 5  # FR-022: fewer than five records -> insufficient-data notice


# ---------- helpers ----------
def _period(t, today):
    """Return (start, end, label) for the time phrase in the message. Default: this month."""
    month_start = today.replace(day=1)
    if re.search(r"النهارده|النهاردة|today|ennharda|elnaharda", t):
        return today, today, "النهارده"
    if re.search(r"امبارح|yesterday|embare7", t):
        d = today - dt.timedelta(days=1)
        return d, d, "امبارح"
    if re.search(r"الاسبوع اللي فات|الاسبوع الماضي|last week", t):
        return today - dt.timedelta(days=13), today - dt.timedelta(days=7), "الأسبوع اللي فات"
    if re.search(r"الاسبوع|اسبوع|this week|week", t):
        return today - dt.timedelta(days=6), today, "آخر 7 أيام"
    if re.search(r"الشهر اللي فات|الشهر الماضي|last month|el shahr elly fat", t):
        end = month_start - dt.timedelta(days=1)
        return end.replace(day=1), end, "الشهر اللي فات"
    if re.search(r"السنه|السنة|this year|year", t):
        return today.replace(month=1, day=1), today, "من أول السنة"
    return month_start, today, "الشهر ده"


def _in_period(entry, start, end):
    try:
        d = dt.date.fromisoformat(entry.get("date", "")[:10])
    except ValueError:
        return False
    return start <= d <= end


def _egp(entry):
    return entry["amount"] if entry.get("currency", "EGP") == "EGP" else 0  # prototype totals count EGP only


def _category_in(t):
    for cat, pattern in CATEGORY_KEYWORDS:
        if cat in ALLOWED["log_expense"] and re.search(pattern, t):
            return cat
    return None


def _month_summary(ledger, today):
    start = today.replace(day=1)
    month = [e for e in ledger if _in_period(e, start, today)]
    by_cat = {}
    for e in month:
        if e["type"] == "expense":
            by_cat[e["category"]] = by_cat.get(e["category"], 0) + _egp(e)
    income = sum(_egp(e) for e in month if e["type"] == "income")
    return month, income, dict(sorted(by_cat.items(), key=lambda kv: -kv[1]))


# ---------- spending_query (FR-018) ----------
def spending_query(text, ledger, today=None):
    today = today or dt.date.today()
    t = normalize(text)
    if not ledger:
        return {"reply": "لسه مفيش معاملات مسجلة عشان أحسبلك. سجّل شوية مصاريف الأول 🙂", "data": {"records": 0}}
    start, end, label = _period(t, today)
    entries = [e for e in ledger if _in_period(e, start, end)]
    if re.search(r"دخل|قبضت|income|earned|مرتب", t):
        total = sum(_egp(e) for e in entries if e["type"] == "income")
        return {"reply": f"دخلك {label}: {_money(total, 'EGP')}.", "data": {"period": label, "income": total}}
    expenses = [e for e in entries if e["type"] == "expense"]
    if re.search(r"اكبر|اكتر|اعلي|most|biggest|top", t):
        by_cat = {}
        for e in expenses:
            by_cat[e["category"]] = by_cat.get(e["category"], 0) + _egp(e)
        if not by_cat:
            return {"reply": f"مفيش مصاريف مسجلة {label}.", "data": {"period": label}}
        cat, amount = max(by_cat.items(), key=lambda kv: kv[1])
        share = amount / sum(by_cat.values())
        return {"reply": f"أكبر بند {label}: {CATEGORY_AR.get(cat, cat)} بـ {_money(amount, 'EGP')} ({share:.0%} من مصاريفك).",
                "data": {"period": label, "top_category": cat, "amount": amount}}
    cat = _category_in(t)
    if cat:
        expenses = [e for e in expenses if e["category"] == cat]
    total = sum(_egp(e) for e in expenses)
    what = f" على {CATEGORY_AR[cat]}" if cat else ""
    if not expenses:
        return {"reply": f"مفيش مصاريف{what} مسجلة {label}.", "data": {"period": label, "category": cat, "total": 0}}
    return {"reply": f"صرفت{what} {label}: {_money(total, 'EGP')} ({len(expenses)} معاملة).",
            "data": {"period": label, "category": cat, "total": total, "count": len(expenses)}}


# ---------- forecast ----------
def forecast(text, ledger, today=None):
    today = today or dt.date.today()
    if len(ledger) < MIN_TRANSACTIONS:
        return {"reply": f"محتاج {MIN_TRANSACTIONS} معاملات على الأقل عشان أتوقع. عندك {len(ledger)} لحد دلوقتي.",
                "data": {"records": len(ledger), "insufficient_data": True}}
    month, income, by_cat = _month_summary(ledger, today)
    spent = sum(by_cat.values())
    days = calendar.monthrange(today.year, today.month)[1]
    projected = spent / today.day * days
    reply = (f"لو فضلت بنفس المعدل، هتصرف حوالي {_money(round(projected, -1), 'EGP')} الشهر ده "
             f"(صرفت {_money(spent, 'EGP')} في {today.day} يوم).")
    if income:
        reply += f" ودخلك المسجل {_money(income, 'EGP')}، يعني هيفضل معاك حوالي {_money(round(income - projected, -1), 'EGP')}."
    if today.day < 7:
        reply += " (التوقع لسه مش دقيق لأننا في أول الشهر.)"
    return {"reply": reply + "\n" + DISCLAIMER,
            "data": {"spent_so_far": spent, "projected_month": round(projected, 2), "income": income, "method": "linear run-rate"}}


# ---------- market (FR-023, NFR-P03) ----------
CURRENCIES = [("USD", r"دولار|usd|\$|dollar|dolar"), ("EUR", r"يورو|euro|eur"), ("SAR", r"ريال|sar|riyal"),
              ("GBP", r"استرليني|باوند|gbp|sterling"), ("AED", r"درهم|aed|dirham"), ("KWD", r"دينار|kwd|dinar")]
OTHER_ASSETS = [("stocks", r"سهم|اسهم|البورصه|egx|stock|share|اسهمي"), ("gold", r"دهب|ذهب|gold|عيار"),
                ("inflation", r"تضخم|inflation"), ("interest", r"فايده|فائده|فوايد|شهاده|شهادات|interest|cbe|المركزي")]
ASSET_AR = {"stocks": "أسعار الأسهم والبورصة", "gold": "أسعار الدهب", "inflation": "أرقام التضخم", "interest": "أسعار الفايدة والشهادات"}
_rates_cache = {"at": 0, "data": None}


def _usd_rates():
    """Live exchange rates from open.er-api.com (free, no key), cached for an hour. None on failure."""
    if _rates_cache["data"] and time.time() - _rates_cache["at"] < 3600:
        return _rates_cache["data"]
    try:
        with urllib.request.urlopen("https://open.er-api.com/v6/latest/USD", timeout=5) as resp:
            data = json.loads(resp.read().decode("utf-8"))
        if data.get("result") != "success" or "EGP" not in data.get("rates", {}):
            return None
        _rates_cache.update(at=time.time(), data=data)
        return data
    except Exception:
        return None


def market(text, ledger=None, today=None):
    t = normalize(text)
    parts, data = [], {"source": None}
    currencies = [code for code, pat in CURRENCIES if re.search(pat, t)]
    assets = [a for a, pat in OTHER_ASSETS if re.search(pat, t)]
    if currencies or not assets:
        rates = _usd_rates()
        if rates:
            egp = rates["rates"]["EGP"]
            for code in currencies or ["USD"]:
                per_unit = egp / rates["rates"][code]
                parts.append(f"1 {code} ≈ {per_unit:,.2f} جنيه")
            updated = rates.get("time_last_update_utc", "")[:16]
            data.update(source="open.er-api.com", updated=updated, currencies=currencies or ["USD"])
            parts.append(f"(المصدر: open.er-api.com، آخر تحديث: {updated})")
        else:
            parts.append("مش قادر أوصل لأسعار الصرف دلوقتي. هكمّل من غير بيانات السوق.")  # NFR-P03 fallback
            data["economic_data_unavailable"] = True
    for a in assets:
        parts.append(f"{ASSET_AR[a]} لسه مش متوصلة في النسخة التجريبية.")
    data["assets_requested"] = assets
    if assets and enabled():
        answer = generate(
            f"سؤال المستخدم: {text}",
            "انت مساعد مالي مصري. اشرح بالعامية المصرية في 3 جمل بالكتير المفاهيم العامة اللي تساعد المستخدم يفهم السؤال. "
            "ممنوع تذكر أي أسعار أو أرقام حالية لأنك مش متأكد منها، وممنوع توصي بشراء أو بيع أصل معين.")
        if answer:
            parts.append(answer)
            data["llm"] = True
    return {"reply": "\n".join(parts) + "\n" + DISCLAIMER, "data": data}


# ---------- advice (FR-022, FR-025, FR-029) ----------
def advice(text, ledger, today=None):
    today = today or dt.date.today()
    month, income, by_cat = _month_summary(ledger or [], today)
    # Only aggregated numbers go to the LLM: no raw messages, merchants or personal details (FR-029).
    summary = {"income_this_month": income, "expenses_by_category_this_month": {CATEGORY_AR.get(k, k): v for k, v in by_cat.items()},
               "transactions_recorded": len(ledger or [])}
    data = {"summary_sent_to_llm": summary, "llm": False}
    answer = None
    if enabled():
        answer = generate(
            f"سؤال المستخدم: {text}\nملخص بياناته (بالجنيه): {json.dumps(summary, ensure_ascii=False)}",
            "انت مساعد مالي مصري لمستخدمين معلوماتهم المالية بسيطة. رد بالعامية المصرية في 4 جمل بالكتير، "
            "بنصايح عملية ومحددة مبنية على الملخص لو مفيد. ماتوصيش بمنتجات استثمار معينة ولا تقول أرقام سوق.")
        data["llm"] = bool(answer)
    if not answer:
        if len(ledger or []) >= MIN_TRANSACTIONS and by_cat:
            cat, amount = next(iter(by_cat.items()))
            share = amount / sum(by_cat.values())
            answer = (f"أكبر بند عندك الشهر ده {CATEGORY_AR.get(cat, cat)} ({share:.0%} من مصاريفك). "
                      f"لو قللته 10% هتوفر حوالي {_money(round(amount * 0.1, -1), 'EGP')} في الشهر. "
                      "وقاعدة بسيطة تبدأ بيها: حوّش 20% من دخلك أول ما تقبض قبل ما تصرف.")
        else:
            answer = "قاعدة بسيطة تبدأ بيها: قسّم دخلك 50% للأساسيات، و30% للمصاريف الشخصية، و20% للتحويش. ولما تسجل مصاريف أكتر، هقدر أديك نصايح على حسب صرفك."
        if not enabled():
            data["note"] = "LLM not configured (set GEMINI_API_KEY); this is the built-in rule-based advice."
    return {"reply": answer + "\n" + DISCLAIMER, "data": data}


# ---------- manage (FR-016) ----------
def manage(text, ledger, today=None):
    t = normalize(text)
    if re.search(r"غير|عدل|edit|change|correct|ghayar|3adel", t):
        return {"reply": "التعديل من الشات جاي في النسخة الجاية. دلوقتي امسح المعاملة من اللوحة وسجلها تاني.", "data": {"action": "edit_not_supported"}}
    if not ledger:
        return {"reply": "مفيش معاملات مسجلة أمسحها.", "data": {}}
    last = ledger[0]
    return {"reply": f"مسحت آخر معاملة: {last.get('label') or CATEGORY_AR.get(last['category'], '')} – {_money(last['amount'], last.get('currency', 'EGP'))} ↩️",
            "data": {"deleted": last}, "action": {"type": "delete", "id": last.get("id")}}


GREETING_REPLY = "أهلاً بيك 👋 أقدر أسجلك مصاريفك ودخلك، وأقولك صرفت كام، وأتوقع مصاريف الشهر، وأديك أسعار العملات ونصايح بسيطة."
OUT_OF_SCOPE_REPLY = "ده برة اللي أقدر أساعد فيه 🙏 أنا متخصص في فلوسك: تسجيل المصاريف، والحسابات، والنصايح، وأسعار العملات."
UNSURE_REPLY = "مش متأكد قصدك 🤔 عاوز تسجل مصروف، ولا تسأل عن مصاريفك، ولا عاوز نصيحة؟"


## 13) الـ Orchestrator — `orchestrator.py`
بيربط كل حاجة: الـ Router بيحدد، والرسالة بتروح للـ Classifier أو الـ Forecast أو الـ APIs/LLM.

In [14]:
"""Top level of the chatbot: route the message first, then hand it to the right component.

  message ──► Router ──► transaction    → Assistant (amount rules + TF-IDF + spaCy)   "Classifier"
                     ├─► spending_query → computed from the user's records
                     ├─► forecast       → run-rate projection                         "Forecast"
                     ├─► market         → exchange-rate API (+ LLM for concepts)      "APIs"
                     ├─► advice         → LLM with an aggregated summary (or rules)   "LLM"
                     ├─► manage         → undo / delete
                     └─► greeting, out_of_scope, unsure → fixed replies
"""


class Orchestrator:
    def __init__(self):
        self.router = Router().load()
        self.assistant = Assistant()

    def handle(self, text, ledger=None, draft=None, category=None):
        ledger = ledger or []
        if draft:  # answering a follow-up question ("كان بكام؟"): stay in the transaction flow
            out = self.assistant.follow_up(draft, text=text, category=category)
            return {**out, "route": "transaction", "route_ar": ROUTE_AR["transaction"], "route_confidence": None}

        r = self.router.predict(text)
        route = r["route"]
        meta = {"route": route, "route_ar": ROUTE_AR[route], "route_confidence": round(r["confidence"], 3)}
        if r["unsure"]:
            return {**meta, "status": "not_applicable", "reply": UNSURE_REPLY, "extracted_data": {"router": r}}
        if route == "transaction":
            out = self.assistant.handle(text)
            if out["status"] == "not_applicable":  # router said transaction but the extractor disagrees
                return {**meta, **out}
            return {**meta, **out}
        if route == "greeting":
            return {**meta, "status": "not_applicable", "reply": GREETING_REPLY, "extracted_data": {}}
        if route == "out_of_scope":
            return {**meta, "status": "not_applicable", "reply": OUT_OF_SCOPE_REPLY, "extracted_data": {}}
        result = globals()[route](text, ledger)
        return {**meta, "status": "answered", "reply": result["reply"], "extracted_data": result["data"],
                "action": result.get("action")}


---
# التجربة

### الداتا

In [15]:
import pandas as pd
synthetic = load_synthetic()
train, test = split(synthetic)
handwritten = load_handwritten()
print(f"synthetic: {len(synthetic)}  train: {len(train)}  test: {len(test)}  hand-written: {len(handwritten)}")
pd.DataFrame(synthetic)[["text", "intent", "amount", "category"]].sample(8, random_state=1)

synthetic: 2000  train: 1599  test: 401  hand-written: 80


,text,intent,amount,category
674,اشتراك يوتيوب بريميوم العائلي ب 175 جنيه,log_expense,175.0,Subscriptions
1699,What are my upcoming bill payments for next week?,question,NaN,NaN
1282,دفعت 780 جنيه قسط الأجهزة في بي تك,log_expense,780.0,Debt
1315,دفعت القسط رقم 6 من قرض البنك ب 3400 جنيه,log_expense,3400.0,Debt
1210,دفعت سهمين في الجمعية مع زمايلي ب 6000 جنيه,log_savings,6000.0,Savings
1636,معايا ١٠٠ ألف جنيه أحطهم في صندوق استثمار ولا ...,question,NaN,NaN
613,عمولة السمسار لما أجرنا الشقة كانت ألفين ونص,log_expense,2500.0,Housing
447,جبت دفاية زيت في الشتا ب ٣١٠٠ جنيه,log_expense,3100.0,Shopping


### المبلغ: الدقة على كل الرسايل اللي فيها مبلغ

In [16]:
tx = [r for r in synthetic if r["amount"] is not None]
ok = sum(extract_amount(r["text"])["amount"] == r["amount"] for r in tx)
print(f"amount accuracy: {ok}/{len(tx)} = {ok / len(tx):.1%}")
pd.DataFrame([{"text": t, "amount": extract_amount(t)["amount"]} for t in
              ["دفعت ميتين جنيه بنزين", "دفعت مصاريف الكورس ألف وخمسمية", "اشتريت 3 تيشيرتات ب 900", "masarif el gam3a 9.5k"]])

amount accuracy: 1591/1591 = 100.0%


,text,amount
0,دفعت ميتين جنيه بنزين,200.0
1,دفعت مصاريف الكورس ألف وخمسمية,1500.0
2,اشتريت 3 تيشيرتات ب 900,900.0
3,masarif el gam3a 9.5k,9500.0


### TF-IDF: تدريب نسخة في الذاكرة (من غير ما نغيّر الموديل المحفوظ) وقياسها

In [17]:
from sklearn.metrics import accuracy_score
import numpy as np
intent_model = _model().fit([normalize(r["text"]) for r in train], [r["intent"] for r in train])
tx_train = [r for r in train if r["intent"] in TX_INTENTS and r["category"]]
category_model = _model().fit([normalize(r["text"]) for r in tx_train], [f"{r['intent']}|{r['category']}" for r in tx_train])
print("intent accuracy:", f"{accuracy_score([r['intent'] for r in test], intent_model.predict([normalize(r['text']) for r in test])):.1%}")
lr = category_model.named_steps["logisticregression"]
names = np.array(category_model.named_steps["tfidfvectorizer"].get_feature_names_out())
pd.DataFrame([{"category": c.split("|")[1], "top character n-grams": " · ".join(list(dict.fromkeys(n.strip() for n in names[np.argsort(lr.coef_[i])[-25:][::-1]] if len(n.strip()) > 1))[:8])}
              for i, c in enumerate(lr.classes_) if c.startswith("log_expense|")])

intent accuracy: 87.3%


,category,top character n-grams
0,Debt,قسط · سط · قس · سد · سددت · سدد · ددت · دد
1,Education,رس · درس · rs · كورس · كتب · ورس · كور · در
2,Entertainment,في · تذ · مع · تذا · اكر · تذاك · ذا · ذاك
3,Food,من · ur · بت · بال · لب · جب · بن · lab
4,Health,شف · كشف · كش · صيدل · صيد · دل · يدلي · صيدلي
5,Housing,شق · شقه · الشق · الشقه · لشقه · لشق · جار · ال
6,Shopping,من · جبت · جب · بت · اي · ون · هد · مول
7,Subscriptions,تراك · راك · شترا · شتراك · اشترا · ترا · جد ·...
8,Transport,لل · lel · le · fi · عربيه · ربيه · عرب · عربي
9,Utilities,نت · شحن · شح · حن · et · فاتور · فاتو · اتور


### spaCy: تحميل الموديل المحفوظ (أو تدريبه من هنا، حوالي 3 دقايق)

In [18]:
TRAIN_MODELS = not (os.path.isdir(SpacyApproach.main_dir) and os.path.exists(TfidfApproach.path))
if TRAIN_MODELS:  # first run: train both models on all 2,000 messages and save them to models/
    TfidfApproach().train(synthetic)
    SpacyApproach().train(synthetic)
ner = SpacyApproach().load()
for t in ["انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه", "طلبت بيتزا من طلبات امبارح ب 320", "اشتريت لاب من مول البستان ب 5000 جنيه"]:
    print(t, "\n   →", [(e.text, e.label_) for e in ner.main(t).ents])

انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه 
   → [('كوتشي نايك', 'ITEM'), ('مول مصر', 'MERCHANT'), ('1200', 'AMOUNT'), ('جنيه', 'CURRENCY')]
طلبت بيتزا من طلبات امبارح ب 320 
   → [('طلبات', 'MERCHANT'), ('320', 'AMOUNT')]
اشتريت لاب من مول البستان ب 5000 جنيه 
   → [('مول البستان', 'MERCHANT'), ('5000', 'AMOUNT'), ('جنيه', 'CURRENCY')]


### الـ pipeline كامل

In [19]:
assistant = Assistant()
rows = []
for t in ["انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه", "اشتريت لاب من مول البستان ب 5000 جنيه",
          "جبت لحمة من عند الجزار ب 450 امبارح", "اشتريت جزمة ب 200 جنيه من محل ابو جزم", "قبضت المرتب ١٥٠٠٠",
          "اشتريت جاكيت جديد", "الحلاق ١٥٠", "صرفت كام على الأكل الشهر ده؟"]:
    out = assistant.handle(t)
    d = out["extracted_data"]
    rows.append({"message": t, "status": out["status"], "amount": d.get("amount"), "category": d.get("category"),
                 "item": d.get("item"), "item_type": d.get("item_type_ar"), "merchant": d.get("merchant"), "reply": out["reply"]})
pd.DataFrame(rows)

,message,status,amount,category,item,item_type,merchant,reply
0,انا اشتريت كوتشي نايك من مول مصر ب 1200 جنيه,confirmed,1200.0,Shopping,كوتشي نايك,أحذية,مول مصر,"اتسجل ✅ مصروف: 1,200 جنيه – مشتريات (كوتشي ناي..."
1,اشتريت لاب من مول البستان ب 5000 جنيه,confirmed,5000.0,Shopping,لاب,إلكترونيات,مول البستان,"اتسجل ✅ مصروف: 5,000 جنيه – مشتريات (لاب من مو..."
2,جبت لحمة من عند الجزار ب 450 امبارح,confirmed,450.0,Shopping,لحمة,بقالة,الجزار,اتسجل ✅ مصروف: 450 جنيه – مشتريات (لحمة من الج...
3,اشتريت جزمة ب 200 جنيه من محل ابو جزم,confirmed,200.0,Shopping,جزمة,أحذية,ابو جزم,اتسجل ✅ مصروف: 200 جنيه – مشتريات (جزمة من ابو...
4,قبضت المرتب ١٥٠٠٠,confirmed,15000.0,Salary,المرتب,NaN,NaN,"اتسجل ✅ دخل: 15,000 جنيه – مرتب (المرتب)"
5,اشتريت جاكيت جديد,needs_info,NaN,Shopping,جاكيت جديد,ملابس,NaN,تمام، جاكيت جديد كان بكام؟
6,الحلاق ١٥٠,confirmed,150.0,Other,NaN,عناية شخصية,NaN,اتسجل ✅ مصروف: 150 جنيه – أخرى
7,صرفت كام على الأكل الشهر ده؟,not_applicable,NaN,NaN,NaN,NaN,NaN,الأسئلة عن مصاريفك جاية في النسخة الجاية 🙏 دلو...


### المقارنة بين الطرق الثلاثة
خلي `RUN_EVALUATE = True` عشان تعيد التدريب والقياس (حوالي 3 دقايق، وبيكتب فوق `models/` بنسخة متدربة على الـ 80% بس، فشغّل خلية التدريب تاني بعدها). غير كده بنعرض آخر نتيجة محفوظة.

In [20]:
from IPython.display import Markdown
RUN_EVALUATE = False
if RUN_EVALUATE:
    evaluate_all()
Markdown(open("results/results.md", encoding="utf-8").read())

Train: 1599 synthetic · Test: 401 synthetic (held out) + 80 hand-written

| Approach | Set | Type | Intent | Amount | Category | End-to-end |
|---|---|---|---|---|---|---|
| A. Rules (keywords + regex) | synthetic test | 78.3% | 76.8% | 100.0% | 44.7% | **46.4%** |
| | hand-written | 92.5% | – | 100.0% | 86.8% | **83.8%** |
| B. TF-IDF + Logistic Regression | synthetic test | 88.0% | 87.3% | 100.0% | 79.6% | **76.8%** |
| | hand-written | 100.0% | – | 100.0% | 92.1% | **92.5%** |
| C. spaCy (NER + textcat) | synthetic test | 88.8% | 88.0% | 99.7% | 71.1% | **72.8%** |
| | hand-written | 97.5% | – | 100.0% | 80.3% | **81.2%** |

| Approach | Model size | Latency / message | Training time |
|---|---|---|---|
| A. Rules (keywords + regex) | 0.0 MB | 0.04 ms | 0.0 s |
| B. TF-IDF + Logistic Regression | 7.23 MB | 0.88 ms | 4.5 s |
| C. spaCy (NER + textcat) | 34.72 MB | 3.31 ms | 146.7 s |

spaCy NER on synthetic test (exact span match): precision 84.2%, recall 79.6%, F1 **81.8%**. Entities skipped in training because of token misalignment: 98.


### الـ Router: كل رسالة رايحة فين؟

In [21]:
router = Router().load()
pd.DataFrame([{"message": t, **router.predict(t)} for t in [
    "اشتريت لاب من مول البستان ب 5000 جنيه", "صرفت كام على الأكل الشهر ده؟", "هصرف كام الشهر ده؟",
    "الدولار بكام النهارده؟", "سهم CIB عامل ايه؟", "احوش ازاي من مرتبي؟", "امسح آخر حاجة", "ازيك", "مين هيكسب الماتش؟"]])

,message,route,confidence,runner_up,unsure
0,اشتريت لاب من مول البستان ب 5000 جنيه,transaction,0.960252,out_of_scope,False
1,صرفت كام على الأكل الشهر ده؟,spending_query,0.982675,forecast,False
2,هصرف كام الشهر ده؟,forecast,0.802650,forecast,False
3,الدولار بكام النهارده؟,market,0.943173,spending_query,False
4,سهم CIB عامل ايه؟,market,0.883032,out_of_scope,False
5,احوش ازاي من مرتبي؟,advice,0.999490,out_of_scope,False
6,امسح آخر حاجة,manage,0.976716,transaction,False
7,ازيك,greeting,0.826205,transaction,False
8,مين هيكسب الماتش؟,out_of_scope,0.454805,forecast,False


In [22]:
Markdown(open("results/router_results.md", encoding="utf-8").read())

Router: trained on 1605, tested on 403 held-out messages. Accuracy **87.3%**

| Route | Correct / total | Accuracy |
|---|---|---|
| transaction (تسجيل معاملة) | 104 / 112 | 92.9% |
| spending_query (سؤال عن مصاريفي) | 50 / 56 | 89.3% |
| forecast (توقع) | 26 / 31 | 83.9% |
| market (أسواق واقتصاد) | 41 / 45 | 91.1% |
| advice (نصيحة) | 36 / 45 | 80.0% |
| manage (تعديل/مسح) | 41 / 42 | 97.6% |
| greeting (تحية) | 30 / 32 | 93.8% |
| out_of_scope (خارج النطاق) | 24 / 40 | 60.0% |

Most common confusions (gold → predicted): out_of_scope → advice (7), out_of_scope → spending_query (5), spending_query → manage (4), transaction → spending_query (3), advice → spending_query (2), advice → out_of_scope (2), advice → market (2), advice → forecast (2), forecast → advice (2), forecast → transaction (2)

Sample errors:

- `el mafrood a2sat el doyoon matzedsh 3an kam % men el moratab?`: gold advice, predicted spending_query
- `اشرحلي طريقة كرة التلج في سداد الديون المتراكمة`: gold advice, predicted out_of_scope
- `المرتب بيتبخر في أول أسبوعين من الشهر ومش فاهم بيروح فين`: gold advice, predicted spending_query
- `Is it better to invest monthly savings in gold bars, money market funds, or bank certificates?`: gold advice, predicted market
- `ashtery 3arabeya mosta3mala cash wala zero b loan?`: gold advice, predicted out_of_scope
- `How can I protect my long-term savings from inflation in Egypt?`: gold advice, predicted market
- `How can I stretch my remaining 2,000 EGP to last until payday in 10 days?`: gold advice, predicted transaction
- `How can I budget effectively when my freelance income fluctuates every month?`: gold advice, predicted forecast
- `ناوي أبطل أطلب دليفري من طلبات الشهر الجاي عشان أوفر`: gold advice, predicted forecast
- `متوسط صرفي اليومي الأسبوع الجاي المفروض ميزيدش عن كام عشان مكسرش الميزانية؟`: gold forecast, predicted advice
- `emta hawصل لهدف ال 100k law ba7awesh b nafs el tari2a?`: gold forecast, predicted advice
- `تفتكر مصاريف الشوبينج الشهر ده هتعدي الشهر اللي فات بقد ايه؟`: gold forecast, predicted spending_query
- `When will all my ValU installments be fully paid off?`: gold forecast, predicted transaction
- `امتى الرصيد بتاعي هينزل تحت الألف جنيه لو مبطلتش طلبات أونلاين؟`: gold forecast, predicted transaction
- `التطبيق ده جامد جدا`: gold greeting, predicted out_of_scope
- `سلام بقى عشان داخل اجتماع في الشغل`: gold greeting, predicted out_of_scope
- `kansel order talabat abo 280 geneh`: gold manage, predicted transaction
- `تصنيف مصر الائتماني الأخير عند وكالات التصنيف اتغير ولا ثابت؟`: gold market, predicted out_of_scope
- `When is the next Central Bank of Egypt monetary policy committee meeting?`: gold market, predicted out_of_scope
- `faydet shahadat el bank el ahly el gedida kam %?`: gold market, predicted spending_query
- `3a2ed ozoon el khazana 12 shahr kam ba3d el dareeba?`: gold market, predicted spending_query
- `kam calorie fe sandwich el shawarma?`: gold out_of_scope, predicted spending_query
- `ازاي انضف بقعة قهوة من على قميص ابيض قطن؟`: gold out_of_scope, predicted advice
- `كام جرام بروتين في ١٠٠ جرام صدور فراخ مشوية؟`: gold out_of_scope, predicted market
- `عايز اغير العملة الأساسية للجنيه المصري`: gold out_of_scope, predicted advice


### الـ Orchestrator كامل (ببيانات مستخدم تجريبية)

In [23]:
import datetime as dt
today = dt.date.today().isoformat()
ledger = [{"id": i, "date": today, "type": t, "amount": a, "currency": "EGP", "category": c, "label": l}
          for i, (t, a, c, l) in enumerate([("expense", 320, "Food", "بيتزا"), ("expense", 1200, "Shopping", "كوتشي"),
                                            ("expense", 150, "Food", "كشري"), ("income", 15000, "Salary", "المرتب"),
                                            ("expense", 200, "Transport", "بنزين")])]
bot = Orchestrator()
for t in ["صرفت كام على الأكل الشهر ده؟", "هصرف كام الشهر ده؟", "الدولار بكام النهارده؟", "احوش ازاي من مرتبي؟", "امسح آخر حاجة"]:
    out = bot.handle(t, ledger=ledger)
    print(f"> {t}\n[{out['route']}] {out['reply']}\n")

> صرفت كام على الأكل الشهر ده؟
[spending_query] صرفت على أكل الشهر ده: 470 جنيه (2 معاملة).

> هصرف كام الشهر ده؟
[forecast] لو فضلت بنفس المعدل، هتصرف حوالي 2,950 جنيه الشهر ده (صرفت 1,870 جنيه في 19 يوم). ودخلك المسجل 15,000 جنيه، يعني هيفضل معاك حوالي 12,050 جنيه.
ℹ️ دي معلومات إرشادية مش استشارة مالية مرخصة.



> الدولار بكام النهارده؟
[market] 1 USD ≈ 53.79 جنيه
(المصدر: open.er-api.com، آخر تحديث: Thu, 19 Nov 2026)
ℹ️ دي معلومات إرشادية مش استشارة مالية مرخصة.

> احوش ازاي من مرتبي؟
[advice] أكبر بند عندك الشهر ده مشتريات (64% من مصاريفك). لو قللته 10% هتوفر حوالي 120 جنيه في الشهر. وقاعدة بسيطة تبدأ بيها: حوّش 20% من دخلك أول ما تقبض قبل ما تصرف.
ℹ️ دي معلومات إرشادية مش استشارة مالية مرخصة.

> امسح آخر حاجة
[manage] مسحت آخر معاملة: بيتزا – 320 جنيه ↩️



### جرّب رسالتك

In [24]:
bot.handle("اشتريت شاحن موبايل من راية ب 450 جنيه", ledger=ledger)["reply"]

'اتسجل ✅ مصروف: 450 جنيه – مشتريات (شاحن موبايل من راية)'